# Corte 3, Punto 3: resiliencia y resultados idempotentes

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T3.1 a T3.5 (checkpoint en MinIO, destino idempotente, deduplicación en el flujo y pruebas de fallo).

Entrada: el flujo del notebook 05.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
import time
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    STREAMING_LOTES,
    STREAMING_ENTRADA,
    STREAMING_TARDIOS,
    STREAMING_CHECKPOINT,
    STREAMING_VENTANAS,
    STREAMING_CHECKPOINT_SIN_REPETIDOS,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.flujo import (
    entregar_lote, vaciar, crear_carpeta, leer_flujo, con_mes_y_departamento, ventanas_mensuales, TablaEnMemoria,
    archivos_por_disparo, lotes_preparados, contenido_checkpoint, leer_texto, copiar_carpeta, DestinoVentanas,
    ventanas_mensuales_sin_repetidos, CLAVE_CONTRATO,
    CLAVES_VENTANA, agregar_por_lotes, registrar_tardios,
)

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T3.1 Punto de control (checkpoint) en MinIO

El flujo del notebook 05 guarda su punto de control en `streaming/checkpoint/ventanas` (`STREAMING_CHECKPOINT` en `src/pamd/config.py`), una ruta de MinIO que solo usa este flujo. La prueba repite la simulación de T2.3 con los mismos lotes y la misma consulta (`src/pamd/flujo.py`), pero **detiene la consulta después del lote 11 y la vuelve a encender con el mismo checkpoint**. Si el checkpoint funciona, al reiniciar no vuelve a leer los 11 lotes ya confirmados y sigue con el 12.

In [2]:
# T3.1 · Parámetros: los mismos del flujo de T2.1 (notebook 05); si se cambian allí, se cambian aquí
TOLERANCIA = "75 days"
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"
PARTICIONES_ESTADO = 4
DISPARO = "2 seconds"
PAUSA_SEGUNDOS = 5
DETENER_TRAS_LOTE = 11  # la consulta se detiene después del lote 11 (el tardío dentro de la tolerancia)

spark.conf.set("spark.sql.shuffle.partitions", PARTICIONES_ESTADO)
orden = lotes_preparados(spark, STREAMING_LOTES)
filas_lote = {r["archivo"]: r["count"] for r in spark.read.parquet(STREAMING_LOTES).groupBy("archivo").count().collect()}
assert len(orden) == len(filas_lote) == 22, "Faltan lotes de T2.2 en MinIO"

tabla = TablaEnMemoria()  # el mismo destino antes y después del reinicio

def encender():
    """La consulta de T2.3 a T2.5, siempre con el mismo checkpoint."""
    return (
        ventanas_mensuales(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
        .writeStream
        .queryName("ventanas_mensuales")
        .outputMode("update")
        .foreachBatch(tabla.guardar)
        .option("checkpointLocation", STREAMING_CHECKPOINT)
        .trigger(processingTime=DISPARO)
        .start()
    )

def entregar(consulta, archivos):
    for i, archivo in enumerate(archivos):
        if i:
            time.sleep(PAUSA_SEGUNDOS)
        entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, archivo)
        consulta.processAllAvailable()
        print(f"{archivo}: entregado y procesado")

def registro(consulta):
    """Un renglón por disparo (el último progreso publicado de cada uno)."""
    por_disparo = {p["batchId"]: p for p in consulta.recentProgress}
    return [
        {"disparo": b, "filas leídas": p["numInputRows"], "marca de agua": p["eventTime"].get("watermark", "")[:10],
         "ventanas en estado": sum(s["numRowsTotal"] for s in p["stateOperators"])}
        for b, p in sorted(por_disparo.items())
    ]

print(f"{len(orden)} lotes preparados; la consulta se detiene tras {orden[DETENER_TRAS_LOTE - 1]}")

22 lotes preparados; la consulta se detiene tras lote_11_tardio_dentro


In [3]:
# T3.1 · Antes del reinicio: simulación limpia hasta el lote 11 y parada de la consulta
vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT, STREAMING_TARDIOS)
crear_carpeta(spark, STREAMING_ENTRADA)

consulta = encender()
entregar(consulta, orden[:DETENER_TRAS_LOTE])
time.sleep(1)
registro_antes = registro(consulta)
ultimo_antes = consulta.lastProgress["batchId"]
consulta.stop()

display(pd.DataFrame(registro_antes))
print(f"Consulta detenida. Último disparo confirmado: {ultimo_antes}")

lote_01_regular: entregado y procesado


lote_02_regular: entregado y procesado


lote_03_regular: entregado y procesado


lote_04_regular: entregado y procesado


lote_05_regular: entregado y procesado


lote_06_regular: entregado y procesado


... (3 outputs left out)

lote_10_regular: entregado y procesado


lote_11_tardio_dentro: entregado y procesado


,disparo,filas leídas,marca de agua,ventanas en estado
0,0,27889,1970-01-01,381
1,1,0,2018-01-16,96
2,2,25620,2018-01-16,388
3,3,0,2018-10-18,96
4,4,28617,2018-10-18,433
5,5,0,2019-08-18,102
6,6,27177,2019-08-18,433
7,7,0,2020-06-18,100
8,8,23720,2020-06-18,332
9,9,0,2021-01-16,101


Consulta detenida. Último disparo confirmado: 20


### Qué guarda el checkpoint

In [4]:
# T3.1 · Contenido del checkpoint después de la parada
archivos_ck = contenido_checkpoint(spark, STREAMING_CHECKPOINT)
QUE_GUARDA = {
    "metadata": "identificador de la consulta",
    "offsets": "un archivo por disparo, escrito ANTES de procesarlo: hasta qué archivo de la fuente llega y la marca de agua",
    "commits": "un archivo por disparo, escrito DESPUÉS de procesarlo: el disparo quedó confirmado",
    "sources": "los archivos de la entrada que la fuente ya encontró, en orden",
    "state": "el estado de la agregación (acumulado de cada ventana abierta), por partición",
}
carpeta = lambda ruta: ruta.split("/")[0]
contenido = pd.DataFrame([
    {"carpeta": c, "archivos": sum(1 for r, _ in archivos_ck if carpeta(r) == c),
     "bytes": sum(t for r, t in archivos_ck if carpeta(r) == c), "qué guarda": QUE_GUARDA.get(c, "")}
    for c in sorted({carpeta(r) for r, _ in archivos_ck})
])
display(contenido)

# Archivos ya leídos, según sources/0 y offsets/
leidos = archivos_por_disparo(spark, STREAMING_CHECKPOINT)
ya_leidos = list(dict.fromkeys(leidos[b] for b in sorted(leidos)))
print(f"Archivos ya leídos ({len(ya_leidos)}):", ", ".join(ya_leidos))

# El último disparo: lo que se planeó (offsets) y su confirmación (commits)
print(f"offsets/{ultimo_antes}:", *leer_texto(spark, f"{STREAMING_CHECKPOINT}/offsets/{ultimo_antes}")[1:], sep="\n  ")
print(f"commits/{ultimo_antes}:", *leer_texto(spark, f"{STREAMING_CHECKPOINT}/commits/{ultimo_antes}")[1:], sep="\n  ")
print(f"Estado: {sum(1 for r, _ in archivos_ck if r.startswith('state/'))} archivos en {PARTICIONES_ESTADO} particiones; "
      f"{registro_antes[-1]['ventanas en estado']} ventanas abiertas al detener la consulta")
assert ya_leidos == orden[:DETENER_TRAS_LOTE], "El checkpoint no registra exactamente los lotes entregados"
assert {"metadata", "offsets", "commits", "sources", "state"} <= set(contenido["carpeta"])

,carpeta,archivos,bytes,qué guarda
0,commits,21,861,"un archivo por disparo, escrito DESPUÉS de pro..."
1,metadata,1,45,identificador de la consulta
2,offsets,21,13975,"un archivo por disparo, escrito ANTES de proce..."
3,sources,11,2129,los archivos de la entrada que la fuente ya en...
4,state,89,152151,el estado de la agregación (acumulado de cada ...


Archivos ya leídos (11): lote_01_regular, lote_02_regular, lote_03_regular, lote_04_regular, lote_05_regular, lote_06_regular, lote_07_regular, lote_08_regular, lote_09_regular, lote_10_regular, lote_11_tardio_dentro
offsets/20:
  {"batchWatermarkMs":1676419200000,"batchTimestampMs":1794869898087,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
  {"logOffset":10}
commits/20:
  {"nextBatchWatermarkMs":1676419200000}
Estado: 89 archivos en 4 particiones; 101 ventana

### Reinicio con el mismo checkpoint

In [5]:
# T3.1 · Después del reinicio: misma consulta, mismo checkpoint
consulta = encender()
consulta.processAllAvailable()  # todavía no ha llegado ningún lote nuevo
time.sleep(1)
leidas_sin_lotes_nuevos = sum(p["numInputRows"] for p in consulta.recentProgress)
print(f"Al reiniciar, sin lotes nuevos, la consulta leyó {leidas_sin_lotes_nuevos} filas")

entregar(consulta, orden[DETENER_TRAS_LOTE:])
time.sleep(1)
registro_despues = registro(consulta)
consulta.stop()
display(pd.DataFrame(registro_despues))

Al reiniciar, sin lotes nuevos, la consulta leyó 0 filas


lote_12_regular: entregado y procesado


lote_13_regular: entregado y procesado


lote_14_regular: entregado y procesado


lote_15_regular: entregado y procesado


lote_16_regular: entregado y procesado


... (3 outputs left out)

lote_20_regular: entregado y procesado


lote_21_regular: entregado y procesado


lote_22_regular: entregado y procesado


,disparo,filas leídas,marca de agua,ventanas en estado
0,21,26337,2023-02-15,236
1,22,0,2023-06-18,102
2,23,23285,2023-06-18,204
3,24,0,2023-09-17,102
4,25,28015,2023-09-17,303
5,26,0,2024-03-18,102
6,27,26354,2024-03-18,238
7,28,0,2024-07-18,102
8,29,25064,2024-07-18,236
9,30,0,2024-11-18,100


### Criterio de aceptación

In [6]:
# T3.1 · Tras detener y reiniciar con el mismo checkpoint, no se releen los lotes confirmados y se sigue con el siguiente
leidos = archivos_por_disparo(spark, STREAMING_CHECKPOINT)
con_datos_antes = [r["disparo"] for r in registro_antes if r["filas leídas"] > 0]
con_datos_despues = [r["disparo"] for r in registro_despues if r["filas leídas"] > 0]
primer_disparo = registro_despues[0]["disparo"]

display(pd.DataFrame([
    {"momento": "antes del reinicio" if b in con_datos_antes else "después del reinicio", "disparo": b,
     "archivo": leidos[b], "filas leídas": r["filas leídas"], "filas del lote": filas_lote[leidos[b]]}
    for b, r in sorted({**{r["disparo"]: r for r in registro_antes}, **{r["disparo"]: r for r in registro_despues}}.items())
    if r["filas leídas"] > 0
]))

assert leidas_sin_lotes_nuevos == 0, "Al reiniciar se volvieron a leer lotes ya confirmados"
assert primer_disparo == ultimo_antes + 1, "La numeración de disparos no continúa después del reinicio"
assert leidos[con_datos_despues[0]] == orden[DETENER_TRAS_LOTE], "Después del reinicio no siguió con el lote siguiente"
assert [leidos[b] for b in con_datos_antes + con_datos_despues] == orden, "Algún lote se leyó dos veces o se saltó"
assert sum(r["filas leídas"] for r in registro_antes + registro_despues) == sum(filas_lote.values()), "Las filas leídas no suman los lotes"
print(f"Antes del reinicio: disparos {con_datos_antes[0]} a {ultimo_antes} ({DETENER_TRAS_LOTE} lotes). "
      f"Después: empieza en el disparo {primer_disparo} sin releer nada y el primer lote que lee es {orden[DETENER_TRAS_LOTE]}.")
print("Criterio de aceptación de T3.1: OK")

,momento,disparo,archivo,filas leídas,filas del lote
0,antes del reinicio,0,lote_01_regular,27889,27889
1,antes del reinicio,2,lote_02_regular,25620,25620
2,antes del reinicio,4,lote_03_regular,28617,28617
3,antes del reinicio,6,lote_04_regular,27177,27177
4,antes del reinicio,8,lote_05_regular,23720,23720
5,antes del reinicio,10,lote_06_regular,30431,30431
6,antes del reinicio,12,lote_07_regular,26812,26812
7,antes del reinicio,14,lote_08_regular,21842,21842
8,antes del reinicio,16,lote_09_regular,34509,34509
9,antes del reinicio,18,lote_10_regular,20887,20887


Antes del reinicio: disparos 0 a 20 (11 lotes). Después: empieza en el disparo 21 sin releer nada y el primer lote que lee es lote_12_regular.
Criterio de aceptación de T3.1: OK


### Cambiar la consulta invalida el checkpoint

El estado guardado tiene la forma de la consulta que lo creó: la clave (`mes`, `departamento`) y los acumulados de cada medida. Si la consulta cambia (otra clave, otras medidas u otra columna de marca de agua), Spark no puede reutilizar ese estado. La prueba se hace sobre una **copia** del checkpoint en `streaming/prueba_cambio/`, que se borra al terminar; el checkpoint real no se toca.

In [7]:
# T3.1 · Prueba: la misma fuente agrupada solo por mes (sin departamento) sobre una copia del checkpoint
PRUEBA = STREAMING_CHECKPOINT.split("/checkpoint")[0] + "/prueba_cambio"
vaciar(spark, PRUEBA)
copiar_carpeta(spark, STREAMING_CHECKPOINT, f"{PRUEBA}/checkpoint")
crear_carpeta(spark, f"{PRUEBA}/entrada")
entregar_lote(spark, STREAMING_LOTES, f"{PRUEBA}/entrada", orden[-1])

consulta_cambiada = (
    con_mes_y_departamento(leer_flujo(spark, f"{PRUEBA}/entrada"), SIN_DEPARTAMENTO)
    .withWatermark("mes", TOLERANCIA)
    .groupBy("mes")
    .agg(F.count("*").alias("contratos"))
    .writeStream.outputMode("update")
    .foreachBatch(lambda df, _: df.collect())
    .option("checkpointLocation", f"{PRUEBA}/checkpoint")
    .start()
)
try:
    consulta_cambiada.processAllAvailable()
    error = ""
except Exception as e:
    error = str(e)
finally:
    consulta_cambiada.stop()
    vaciar(spark, PRUEBA)

assert "StateSchemaNotCompatible" in error, "Se esperaba que la consulta cambiada no pudiera usar el checkpoint"
lineas_clave = list(dict.fromkeys(l.strip() for l in error.splitlines() if "key schema" in l))
print(*lineas_clave[:2], sep="\n")
print("La consulta cambiada se detuvo con StateSchemaNotCompatible: el estado guardado no tiene su forma.")

- Provided key schema: StructType(StructField(mes,TimestampType,true))
- Existing key schema: StructType(StructField(mes,TimestampType,true),StructField(departamento,StringType,true))
La consulta cambiada se detuvo con StateSchemaNotCompatible: el estado guardado no tiene su forma.


**Qué queda documentado para T3.5.**
- El checkpoint guarda qué archivos ya se leyeron (`sources`), hasta dónde llegó cada disparo y con qué marca de agua (`offsets`), qué disparos quedaron confirmados (`commits`) y el acumulado de las ventanas abiertas (`state`). Al reiniciar, Spark retoma desde el último disparo confirmado: si se cayó después de escribir `offsets` y antes de `commits`, repite ese disparo con exactamente los mismos archivos.
- Cambiar la consulta puede invalidar el checkpoint: otra clave o medidas distintas fallan con `StateSchemaNotCompatible` (prueba de arriba). Otros parámetros quedan fijados en el checkpoint y se ignoran si se cambian después, como `spark.sql.shuffle.partitions` (aparece en `offsets`).
- Borrar el checkpoint hace que el flujo vuelva a leer todos los archivos de la entrada desde cero.
- El checkpoint protege lo que Spark lee y su estado, no el destino: la tabla en memoria de T2 vive en el kernel. Si se reinicia el kernel, Spark retoma bien, pero las ventanas que ya se habían emitido no están en el destino. Por eso el destino idempotente en MinIO de T3.2 es la otra mitad del mecanismo.

## T3.2 Destino idempotente

El checkpoint de T3.1 protege lo que Spark lee y su estado, pero no el destino. Si el proceso se cae después de escribir un micro-lote y antes de confirmarlo en `commits`, Spark lo repite al reiniciar. Un destino que añade filas (`append`) contaría ese micro-lote dos veces; por eso el destino debe **reemplazar**.

`DestinoVentanas` (`src/pamd/flujo.py`) guarda la tabla de ventanas en `streaming/ventanas` (`STREAMING_VENTANAS`) con dos escrituras por micro-lote, y las dos reemplazan:

1. `cambios/batch_id=N`: las ventanas que cambió el micro-lote N (en modo `update` solo llegan esas, con su acumulado). La carpeta se sobrescribe entera.
2. `tabla/anio_mes=AAAA-MM`: la tabla que se consulta. Para cada mes tocado por el micro-lote se toma, de todos los cambios hasta N, la última versión de cada ventana, y se sobrescribe la partición de ese mes (sobrescritura dinámica de particiones: los demás meses no se tocan).

La tabla se rehace siempre a partir de `cambios/`, no a partir de sí misma. Un lote tardío trae solo algunos departamentos de un mes; si la partición del mes se rehiciera con lo que llega más lo que ya había en la tabla, una caída entre el borrado y la escritura de la partición perdería los departamentos que el lote no traía.

No se usa `MERGE` porque el entorno no tiene un formato de tablas transaccional (Delta o Iceberg); el destino es Parquet.

### El flujo completo con el destino en MinIO

La misma consulta de T2.3 a T2.5, en una simulación limpia. Cada micro-lote se guarda en los dos destinos, la tabla en memoria de T2 y la tabla en MinIO, para compararlos al final.

In [8]:
# T3.2 · Flujo completo: cada micro-lote va a la tabla en memoria (T2) y al destino idempotente en MinIO
memoria = TablaEnMemoria()
destino = DestinoVentanas(spark, STREAMING_VENTANAS)
micro_lotes = {}  # las ventanas que trajo cada micro-lote, para poder repetirlo después

def guardar_en_los_dos(df, batch_id):
    df.persist()
    try:
        micro_lotes[batch_id] = df.collect()
        memoria.guardar(df, batch_id)
        destino.guardar(df, batch_id)
    finally:
        df.unpersist()

vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT, STREAMING_TARDIOS, STREAMING_VENTANAS)
crear_carpeta(spark, STREAMING_ENTRADA)

inicio = perf_counter()
consulta = (
    ventanas_mensuales(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
    .writeStream
    .queryName("ventanas_mensuales_minio")
    .outputMode("update")
    .foreachBatch(guardar_en_los_dos)
    .option("checkpointLocation", STREAMING_CHECKPOINT)
    .trigger(processingTime=DISPARO)
    .start()
)
entregar(consulta, orden)
consulta.stop()
print(f"Flujo terminado en {perf_counter() - inicio:.0f} s; {sum(1 for f in micro_lotes.values() if f)} micro-lotes con ventanas cambiadas")

lote_01_regular: entregado y procesado


lote_02_regular: entregado y procesado


lote_03_regular: entregado y procesado


lote_04_regular: entregado y procesado


lote_05_regular: entregado y procesado


lote_06_regular: entregado y procesado


... (12 outputs left out)

lote_19_regular: entregado y procesado


lote_20_regular: entregado y procesado


lote_21_regular: entregado y procesado


lote_22_regular: entregado y procesado
Flujo terminado en 215 s; 21 micro-lotes con ventanas cambiadas


### La tabla en MinIO frente a la tabla en memoria

Las cifras de cada tabla son el número de ventanas, el total de contratos, el valor total (normal más atípicos) y una huella (suma del `xxhash64` de cada fila). Dos tablas con las mismas cuatro cifras tienen las mismas filas; además se comprueba fila a fila con `exceptAll`.

In [9]:
# T3.2 · Cifras de una tabla de ventanas, y comparación de los dos destinos de la misma ejecución
def cifras(df):
    r = df.agg(
        F.count("*").alias("ventanas"),
        F.sum("contratos").alias("contratos"),
        F.sum(F.col("valor_total") + F.col("valor_atipicos")).alias("valor"),
        F.sum(F.xxhash64(*df.columns).cast("decimal(38,0)")).alias("huella"),
    ).first()
    return {"ventanas": r["ventanas"], "contratos": r["contratos"], "valor": f"{r['valor']:,.2f}", "huella": str(r["huella"])}

en_minio, en_memoria = destino.leer(), memoria.como_dataframe(spark)
cifras_minio, cifras_memoria = cifras(en_minio), cifras(en_memoria)
display(pd.DataFrame([{"destino": "tabla en MinIO (T3.2)", **cifras_minio}, {"destino": "tabla en memoria (T2)", **cifras_memoria}]))

archivos_destino = contenido_checkpoint(spark, STREAMING_VENTANAS)
carpetas = lambda prefijo: len({r.split("/")[1] for r, _ in archivos_destino if r.startswith(prefijo + "/") and "=" in r})
print(f"En MinIO: {carpetas('cambios')} carpetas en cambios/ (una por micro-lote con cambios) y {carpetas('tabla')} particiones en tabla/ (una por mes)")

assert cifras_minio == cifras_memoria, "La tabla en MinIO no coincide con la tabla en memoria"
assert en_minio.exceptAll(en_memoria).count() == 0 and en_memoria.exceptAll(en_minio).count() == 0, "Hay filas distintas"
print("La tabla en MinIO tiene exactamente las mismas filas que la tabla en memoria.")

,destino,ventanas,contratos,valor,huella
0,tabla en MinIO (T3.2),3735,529326,"175,067,420,085,034,807.00",358729028860944948943
1,tabla en memoria (T2),3735,529326,"175,067,420,085,034,807.00",358729028860944948943


En MinIO: 21 carpetas en cambios/ (una por micro-lote con cambios) y 136 particiones en tabla/ (una por mes)


La tabla en MinIO tiene exactamente las mismas filas que la tabla en memoria.


### El mismo micro-lote una, dos y diez veces

Se repite el último micro-lote con datos, que es el que Spark repetiría tras una caída (los anteriores ya están confirmados en el checkpoint). Se llama a `destino.guardar` con el mismo contenido y el mismo número de micro-lote, y se calculan las cifras de la tabla cuando se ha guardado una, dos y diez veces.

In [10]:
# T3.2 · Repetir el último micro-lote: las cifras de la tabla no cambian
ultimo = max(b for b, filas in micro_lotes.items() if filas)
micro_lote = spark.createDataFrame(micro_lotes[ultimo], memoria.esquema)
r = micro_lote.agg(F.count("*").alias("v"), F.sum("contratos").alias("c")).first()
print(f"Micro-lote {ultimo}: trae {r['v']} ventanas que suman {r['c']:,} contratos")

pruebas, veces = [{"veces guardado": 1, **cifras(destino.leer())}], 1  # la primera vez fue durante el flujo
for objetivo in (2, 10):
    while veces < objetivo:
        destino.guardar(micro_lote, ultimo)
        veces += 1
    pruebas.append({"veces guardado": veces, **cifras(destino.leer())})
display(pd.DataFrame(pruebas))

sin_veces = [{k: v for k, v in p.items() if k != "veces guardado"} for p in pruebas]
assert sin_veces[0] == sin_veces[1] == sin_veces[2] == cifras_memoria, "Repetir el micro-lote cambió las cifras"
print("Guardar el mismo micro-lote 1, 2 o 10 veces deja las mismas cifras.")

Micro-lote 40: trae 103 ventanas que suman 20,913 contratos


,veces guardado,ventanas,contratos,valor,huella
0,1,3735,529326,"175,067,420,085,034,807.00",358729028860944948943
1,2,3735,529326,"175,067,420,085,034,807.00",358729028860944948943
2,10,3735,529326,"175,067,420,085,034,807.00",358729028860944948943


Guardar el mismo micro-lote 1, 2 o 10 veces deja las mismas cifras.


### Contraste: el mismo micro-lote en modo `append`

La misma prueba sobre una copia de la tabla en una carpeta aparte, pero añadiendo el micro-lote en vez de reemplazar. Con `append` cada repetición suma otra vez las ventanas del micro-lote, y ya la primera las duplica, porque esas ventanas estaban en la tabla.

In [11]:
# T3.2 · Contraste con append sobre una copia de la tabla (la carpeta de prueba se borra al final)
PRUEBA_APPEND = STREAMING_VENTANAS + "_prueba_append"
vaciar(spark, PRUEBA_APPEND)
destino.leer().write.mode("overwrite").parquet(PRUEBA_APPEND)

def cifras_append():
    c = cifras(spark.read.parquet(PRUEBA_APPEND))
    return {"filas": c["ventanas"], "contratos": c["contratos"], "valor": c["valor"]}

con_append, veces = [{"veces añadido": 0, **cifras_append()}], 0
for objetivo in (1, 2, 10):
    while veces < objetivo:
        micro_lote.write.mode("append").parquet(PRUEBA_APPEND)
        veces += 1
    con_append.append({"veces añadido": veces, **cifras_append()})
vaciar(spark, PRUEBA_APPEND)
display(pd.DataFrame(con_append))

assert [p["contratos"] for p in con_append] == [con_append[0]["contratos"] + n * r["c"] for n in (0, 1, 2, 10)], "append no sumó lo esperado"
print(f"Con append, cada repetición añade {r['v']} filas y {r['c']:,} contratos que ya estaban contados.")
print("Criterio de aceptación de T3.2: OK")

,veces añadido,filas,contratos,valor
0,0,3735,529326,"175,067,420,085,034,807.00"
1,1,3838,550239,"183,142,906,911,071,778.00"
2,2,3941,571152,"191,218,393,737,108,749.00"
3,10,4765,738456,"255,822,288,345,404,517.00"


Con append, cada repetición añade 103 filas y 20,913 contratos que ya estaban contados.
Criterio de aceptación de T3.2: OK


**Qué queda documentado para T3.5.**
- El destino reemplaza en vez de sumar: cada micro-lote sobrescribe su carpeta en `cambios/` y las particiones de los meses que toca en `tabla/`. Repetirlo deja la tabla igual; con `append` las cifras crecen en cada repetición.
- Si el flujo se cae mientras escribe, el micro-lote no queda confirmado en el checkpoint y Spark lo repite al reiniciar con los mismos datos. La repetición rehace las dos escrituras completas, así que no importa en qué punto se cortó.
- Límite: la idempotencia cubre la repetición del último micro-lote, que es la única que hace Spark. Al guardar el micro-lote N, el destino borra los cambios con número mayor, porque solo pueden ser restos de una ejecución anterior; guardar a mano un micro-lote antiguo descarta los posteriores.
- Límite: si se borra el checkpoint, la numeración de micro-lotes vuelve a empezar. Repetir la misma entrada sobre el mismo destino da la misma tabla (T3.4, escenario B); si la entrada es otra, hay que vaciar `streaming/ventanas`, porque los meses que la nueva entrada no toque conservarían las cifras anteriores.
- El destino no evita que un archivo repetido en la entrada cuente dos veces: para Spark es un archivo nuevo. Eso lo resuelve la deduplicación de T3.3.

## T3.3 Deduplicación dentro del flujo

El checkpoint evita que Spark relea un archivo y el destino de T3.2 evita que un micro-lote repetido cuente dos veces. Queda un caso: que el mismo lote se deposite otra vez en la carpeta de entrada **con otro nombre**. Para Spark es un archivo nuevo, lo lee y lo suma.

La defensa es identificar cada contrato por la clave natural de T1.1 (`id_del_proceso`, `id_adjudicacion`; `CLAVE_CONTRATO` en `src/pamd/flujo.py`) y descartar dentro del flujo los que ya pasaron: `dropDuplicates` con marca de agua, antes de la agregación (`ventanas_mensuales_sin_repetidos`). Spark guarda en el estado las claves de los meses abiertos y las libera cuando el mes queda por debajo de la marca de agua.

### Sin deduplicación, el lote repetido cuenta dos veces

La consulta de T3.2 se reinicia con su checkpoint y recibe otra vez el último lote con otro nombre.

In [12]:
# T3.3 · Contraste: la consulta sin deduplicación (la de T3.2, con su checkpoint) recibe el último lote con otro nombre
REPETIDO = orden[-1]
claves = spark.read.parquet(STREAMING_LOTES).agg(F.count("*").alias("filas"), F.countDistinct(*CLAVE_CONTRATO).alias("claves")).first()
assert claves["filas"] == claves["claves"], "La clave de T1.1 no es única en los lotes"
print(f"Los lotes tienen {claves['filas']:,} contratos y {claves['claves']:,} claves distintas: la clave identifica cada contrato")

sin_dedup = TablaEnMemoria()  # copia del resultado de T3.2, para no tocar el original
sin_dedup.filas, sin_dedup.esquema = memoria.copia(), memoria.esquema
antes = cifras(sin_dedup.como_dataframe(spark))

consulta = (
    ventanas_mensuales(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
    .writeStream
    .queryName("ventanas_sin_deduplicar")
    .outputMode("update")
    .foreachBatch(sin_dedup.guardar)
    .option("checkpointLocation", STREAMING_CHECKPOINT)
    .trigger(processingTime=DISPARO)
    .start()
)
entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, REPETIDO, nombre=REPETIDO + "_copia")
consulta.processAllAvailable()
consulta.stop()
despues = cifras(sin_dedup.como_dataframe(spark))

display(pd.DataFrame([{"momento": "antes de repetir el lote", **antes}, {"momento": f"después de {REPETIDO}_copia", **despues}]).drop(columns="huella"))
contados_dos_veces = despues["contratos"] - antes["contratos"]
assert contados_dos_veces > 0, "Se esperaba que el lote repetido cambiara las cifras"
print(f"Sin deduplicación, {contados_dos_veces:,} contratos de {REPETIDO} se contaron dos veces "
      f"(el resto del lote, {filas_lote[REPETIDO] - contados_dos_veces:,}, es de un mes ya cerrado por la marca de agua).")

Los lotes tienen 530,192 contratos y 530,192 claves distintas: la clave identifica cada contrato


,momento,ventanas,contratos,valor
0,antes de repetir el lote,3735,529326,"175,067,420,085,034,807.00"
1,después de lote_22_regular_copia,3735,542505,"183,136,444,123,501,482.00"


Sin deduplicación, 13,179 contratos de lote_22_regular se contaron dos veces (el resto del lote, 7,734, es de un mes ya cerrado por la marca de agua).


### El flujo completo con deduplicación

Simulación limpia con la consulta nueva, su propio checkpoint (`STREAMING_CHECKPOINT_SIN_REPETIDOS`) y el destino de T3.2. Sin lotes repetidos, el resultado debe ser el mismo de T3.2. La consulta queda encendida para las pruebas siguientes.

In [13]:
# T3.3 · Flujo completo con deduplicación por la clave de T1.1, escribiendo en el destino idempotente
destino = DestinoVentanas(spark, STREAMING_VENTANAS)
vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT_SIN_REPETIDOS, STREAMING_VENTANAS)
crear_carpeta(spark, STREAMING_ENTRADA)

def encender_sin_repetidos():
    return (
        ventanas_mensuales_sin_repetidos(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
        .writeStream
        .queryName("ventanas_sin_repetidos")
        .outputMode("update")
        .foreachBatch(destino.guardar)
        .option("checkpointLocation", STREAMING_CHECKPOINT_SIN_REPETIDOS)
        .trigger(processingTime=DISPARO)
        .start()
    )

inicio = perf_counter()
consulta = encender_sin_repetidos()
entregar(consulta, orden)
time.sleep(1)
cifras_sin_repetidos = cifras(destino.leer())
display(pd.DataFrame([{"consulta": "con deduplicación (T3.3)", **cifras_sin_repetidos}, {"consulta": "sin deduplicación (T3.2)", **cifras_memoria}]))
assert cifras_sin_repetidos == cifras_memoria, "La deduplicación cambió el resultado de una ejecución sin repetidos"
print(f"Flujo terminado en {perf_counter() - inicio:.0f} s. Sin lotes repetidos, la deduplicación no cambia el resultado.")

lote_01_regular: entregado y procesado


lote_02_regular: entregado y procesado


lote_03_regular: entregado y procesado


lote_04_regular: entregado y procesado


lote_05_regular: entregado y procesado


lote_06_regular: entregado y procesado


... (14 outputs left out)

lote_21_regular: entregado y procesado


lote_22_regular: entregado y procesado


,consulta,ventanas,contratos,valor,huella
0,con deduplicación (T3.3),3735,529326,"175,067,420,085,034,807.00",358729028860944948943
1,sin deduplicación (T3.2),3735,529326,"175,067,420,085,034,807.00",358729028860944948943


Flujo terminado en 217 s. Sin lotes repetidos, la deduplicación no cambia el resultado.


### Volver a depositar un lote ya procesado

Tres casos sobre la consulta encendida:

1. **El mismo archivo, con el mismo nombre.** La fuente ya lo tiene anotado en el checkpoint (`sources`) y no lo lee.
2. **El último lote con otro nombre.** Sus meses siguen abiertos: Spark lo lee y la deduplicación descarta los contratos que ya vio.
3. **Un lote antiguo con otro nombre.** Sus meses ya se cerraron y sus claves se liberaron del estado; las filas llegan por debajo de la marca de agua y se descartan por tardías.

In [14]:
# T3.3 · Los tres casos: filas leídas, por qué se descartan y cifras de la tabla después de cada uno
def descartes(p):
    repetidas = sum(s.get("customMetrics", {}).get("numDroppedDuplicateRows", s.get("numDroppedDuplicateRows", 0)) for s in p["stateOperators"])
    tardias = sum(s.get("numRowsDroppedByWatermark", 0) for s in p["stateOperators"])
    return repetidas, tardias

def con_datos():
    return {p["batchId"]: p for p in consulta.recentProgress if p["numInputRows"] > 0}

ANTIGUO = orden[4]
CASOS = [
    ("1. mismo nombre", REPETIDO, None),
    ("2. otro nombre, meses abiertos", REPETIDO, REPETIDO + "_copia"),
    ("3. otro nombre, meses cerrados", ANTIGUO, ANTIGUO + "_copia"),
]
resultados = []
for caso, archivo, nombre in CASOS:
    vistos = set(con_datos())
    entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, archivo, nombre=nombre)
    consulta.processAllAvailable()
    time.sleep(1)
    nuevos = [p for b, p in con_datos().items() if b not in vistos]
    leidas = sum(p["numInputRows"] for p in nuevos)
    repetidas = sum(descartes(p)[0] for p in nuevos)
    tardias = sum(descartes(p)[1] for p in nuevos)
    c = cifras(destino.leer())
    resultados.append({
        "caso": caso, "archivo depositado": f"{nombre or archivo}.parquet", "filas del lote": filas_lote[archivo],
        "filas leídas": leidas, "descartadas por repetidas": repetidas, "descartadas por tardías": tardias,
        "ventanas": c["ventanas"], "contratos": c["contratos"], "valor": c["valor"],
        "cifras": "iguales" if c == cifras_sin_repetidos else "DISTINTAS",
    })
consulta.stop()
display(pd.DataFrame([{"caso": "antes de las pruebas", **{k: cifras_sin_repetidos[k] for k in ("ventanas", "contratos", "valor")}}]))
display(pd.DataFrame(resultados))

assert all(r["cifras"] == "iguales" for r in resultados), "Un lote repetido cambió las cifras"
assert resultados[0]["filas leídas"] == 0, "La fuente volvió a leer un archivo con el mismo nombre"
for r in resultados[1:]:
    assert r["filas leídas"] == r["filas del lote"], "El lote con otro nombre no se leyó completo"
    assert r["descartadas por repetidas"] + r["descartadas por tardías"] == r["filas leídas"], "Alguna fila repetida no se descartó"
assert resultados[1]["descartadas por repetidas"] > 0 and resultados[2]["descartadas por repetidas"] == 0
print("Criterio de aceptación de T3.3: OK (volver a depositar un lote, con el mismo nombre o con otro, no cambia las cifras)")

,caso,ventanas,contratos,valor
0,antes de las pruebas,3735,529326,"175,067,420,085,034,807.00"


,caso,archivo depositado,filas del lote,filas leídas,descartadas por repetidas,descartadas por tardías,ventanas,contratos,valor,cifras
0,1. mismo nombre,lote_22_regular.parquet,20913,0,0,0,3735,529326,"175,067,420,085,034,807.00",iguales
1,"2. otro nombre, meses abiertos",lote_22_regular_copia.parquet,20913,20913,13179,7734,3735,529326,"175,067,420,085,034,807.00",iguales
2,"3. otro nombre, meses cerrados",lote_05_regular_copia.parquet,23720,23720,0,23720,3735,529326,"175,067,420,085,034,807.00",iguales


Criterio de aceptación de T3.3: OK (volver a depositar un lote, con el mismo nombre o con otro, no cambia las cifras)


**Qué queda documentado para T3.5.**
- Un contrato se identifica por `id_del_proceso` + `id_adjudicacion` (la clave de T1.1), que es única en gold y en los lotes.
- Hay tres barreras, una por cada forma de repetición: el checkpoint (un archivo con el mismo nombre no se vuelve a leer), la deduplicación por clave (un archivo con otro nombre cuyos meses siguen abiertos) y la marca de agua (un archivo con otro nombre cuyos meses ya se cerraron). El destino idempotente cubre la cuarta: que Spark repita un micro-lote.
- Límite: la deduplicación solo recuerda las claves de los meses abiertos (la tolerancia de 75 días). Más atrás, lo que protege es la marca de agua, que descarta la fila por tardía sin mirar si es repetida. Si las filas tardías se guardan aparte para corregir las ventanas (T2.5), esa tabla debe deduplicarse por la misma clave antes de sumarla.
- Límite: la consulta con deduplicación tiene un operador con estado más que la de T3.1, así que no puede reutilizar su checkpoint; usa `streaming/checkpoint/ventanas_sin_repetidos`.
- La deduplicación compara la clave, no el contenido: si llega una versión corregida de un contrato ya visto, se descarta. Las correcciones se resuelven aguas arriba, en silver (T1.2).

## T3.4 Pruebas de fallo

El mecanismo completo (checkpoint de T3.1, destino idempotente de T3.2 y deduplicación de T3.3) se somete a tres escenarios. El resultado de cada uno se compara, ventana por ventana, con dos referencias:

- **La ejecución limpia:** el flujo completo, de una vez y sin interrupciones.
- **El cálculo por lotes:** la misma agregación con una consulta normal (como en T2.6) sobre los contratos que el flujo acepta, es decir, todos menos los que llegaron fuera de la tolerancia y quedaron registrados en `streaming/tardios/`.

| Escenario | Qué se hace |
|---|---|
| A | Se detiene la consulta a mitad y se reinicia; más adelante el proceso se cae **mientras escribe** un micro-lote y se reinicia otra vez |
| B | El flujo completo se ejecuta dos veces seguidas sobre el mismo destino |
| C | Se vuelve a depositar un lote ya procesado (con el mismo nombre y con otro) |

Los lotes se entregan con una pausa de 1 segundo en vez de 5, para acortar las cuatro ejecuciones; el orden de llegada es el mismo.

In [15]:
# T3.4 · Herramientas comunes a los escenarios
PAUSA_PRUEBAS = 1
MEDIDAS = ["contratos", "contratos_atipicos", "contratos_valor_cero", "valor_total", "valor_atipicos"]
destino = DestinoVentanas(spark, STREAMING_VENTANAS)

def encender_final(guardar):
    """La consulta definitiva: deduplicación por clave, ventanas mensuales y destino en MinIO."""
    return (
        ventanas_mensuales_sin_repetidos(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
        .writeStream
        .queryName("ventanas_pruebas_de_fallo")
        .outputMode("update")
        .foreachBatch(guardar)
        .option("checkpointLocation", STREAMING_CHECKPOINT_SIN_REPETIDOS)
        .trigger(processingTime=DISPARO)
        .start()
    )

def empezar_de_cero(conservar_destino=False):
    vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT_SIN_REPETIDOS, *([] if conservar_destino else [STREAMING_VENTANAS]))
    crear_carpeta(spark, STREAMING_ENTRADA)

def disparos_con_datos(consulta):
    return {p["batchId"]: p for p in consulta.recentProgress if p["numInputRows"] > 0}

def entregar_lotes(consulta, archivos, registrar_tardias=False):
    """Entrega los lotes uno a uno. Con registrar_tardias guarda aparte las filas descartadas por la marca de agua (T2.5)."""
    tardias = 0
    for i, archivo in enumerate(archivos):
        if i:
            time.sleep(PAUSA_PRUEBAS)
        vistos = set(disparos_con_datos(consulta))
        entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, archivo)
        consulta.processAllAvailable()
        if registrar_tardias:
            nuevos = []
            for _ in range(20):  # el progreso se publica justo después de confirmar el micro-lote
                nuevos = [p for b, p in disparos_con_datos(consulta).items() if b not in vistos]
                if nuevos:
                    break
                time.sleep(0.5)
            assert len(nuevos) == 1, f"{archivo}: se esperaba un disparo con datos y hubo {len(nuevos)}"
            p = nuevos[0]
            if sum(s.get("numRowsDroppedByWatermark", 0) for s in p["stateOperators"]):
                tardias += registrar_tardios(spark, STREAMING_ENTRADA, archivo, p["eventTime"]["watermark"], STREAMING_TARDIOS, SIN_DEPARTAMENTO)
    return tardias

def foto():
    """Copia en el driver de la tabla del destino, para compararla después aunque el destino cambie."""
    t = destino.leer()
    return spark.createDataFrame(t.collect(), t.schema)

def ventanas_distintas(a, b):
    """Número de ventanas que están en una sola tabla o tienen alguna medida distinta."""
    distinta = F.col("a.contratos").isNull() | F.col("b.contratos").isNull()
    for m in MEDIDAS:
        distinta = distinta | (F.coalesce(F.col(f"a.{m}"), F.lit(0)) != F.coalesce(F.col(f"b.{m}"), F.lit(0)))
    return a.alias("a").join(b.alias("b"), CLAVES_VENTANA, "full_outer").filter(distinta).count()

tablas = {}  # la tabla final de cada escenario

### Referencias: ejecución limpia y cálculo por lotes

La ejecución limpia registra en `streaming/tardios/` las filas que la marca de agua descarta, como en T2.5. El cálculo por lotes agrega, con una consulta normal, los contratos de los lotes que no están en esa tabla.

In [16]:
# T3.4 · Ejecución limpia: el flujo completo, de una vez y sin interrupciones
vaciar(spark, STREAMING_TARDIOS)
empezar_de_cero()
inicio = perf_counter()
consulta = encender_final(destino.guardar)
filas_tardias = entregar_lotes(consulta, orden, registrar_tardias=True)
consulta.stop()
tablas["limpia"] = foto()
print(f"Ejecución limpia en {perf_counter() - inicio:.0f} s; {filas_tardias:,} filas fuera de la tolerancia registradas en streaming/tardios/")

# Cálculo por lotes sobre los contratos aceptados (todos menos los tardíos registrados)
lotes_df = spark.read.parquet(STREAMING_LOTES).drop("archivo")
tardios_df = spark.read.parquet(STREAMING_TARDIOS).select(*CLAVE_CONTRATO).distinct()
aceptados = lotes_df.join(tardios_df, CLAVE_CONTRATO, "left_anti")
por_lotes = agregar_por_lotes(aceptados, SIN_DEPARTAMENTO)
tablas["por lotes"] = spark.createDataFrame(por_lotes.collect(), por_lotes.schema)

total, n_tardios, n_aceptados = lotes_df.count(), tardios_df.count(), aceptados.count()
print(f"Contratos en los lotes: {total:,} = {n_aceptados:,} aceptados por el flujo + {n_tardios:,} tardíos registrados aparte")
dif = ventanas_distintas(tablas["limpia"], tablas["por lotes"])
display(pd.DataFrame([{"tabla": "ejecución limpia (flujo)", **cifras(tablas["limpia"])}, {"tabla": "cálculo por lotes", **cifras(tablas["por lotes"])}]).drop(columns="huella"))
assert n_tardios == filas_tardias and n_aceptados + n_tardios == total
assert dif == 0, f"La ejecución limpia difiere del cálculo por lotes en {dif} ventanas"
print("La ejecución limpia coincide con el cálculo por lotes en todas las ventanas.")

Ejecución limpia en 129 s; 866 filas fuera de la tolerancia registradas en streaming/tardios/


Contratos en los lotes: 530,192 = 529,326 aceptados por el flujo + 866 tardíos registrados aparte


,tabla,ventanas,contratos,valor
0,ejecución limpia (flujo),3735,529326,"175,067,420,085,034,807.00"
1,cálculo por lotes,3735,529326,"175,067,420,085,034,807.00"


La ejecución limpia coincide con el cálculo por lotes en todas las ventanas.


### Escenario A: detener a mitad, reiniciar y caída durante la escritura

Dos interrupciones en la misma ejecución:

1. **Parada ordenada** tras el lote 8 (`consulta.stop()`) y reinicio con el mismo checkpoint, como en T3.1.
2. **Caída mientras escribe** el lote 15: el destino escribe el paso 1 (la carpeta de `cambios/`) y el proceso falla antes del paso 2 (la tabla). El micro-lote no queda confirmado en el checkpoint. Es el peor momento posible: el destino queda a medias.

Después de cada interrupción se enciende otra vez la misma consulta y se entregan los lotes que faltan.

In [17]:
# T3.4 · Escenario A: parada ordenada tras el lote 8 y caída a mitad de la escritura del lote 15
PARADA, CAIDA = 8, 15
empezar_de_cero()

consulta = encender_final(destino.guardar)
entregar_lotes(consulta, orden[:PARADA])
consulta.stop()
print(f"Parada ordenada tras {orden[PARADA - 1]}: {cifras(destino.leer())['contratos']:,} contratos en la tabla")

caida = {"activa": False, "micro_lote": None}
def guardar_con_caida(df, batch_id):
    if caida["activa"] and not df.isEmpty():
        destino._escribir_cambios(df, batch_id)  # paso 1 hecho; el paso 2 (la tabla) no llega a ejecutarse
        caida.update(activa=False, micro_lote=batch_id)
        raise RuntimeError("Caída simulada a mitad de la escritura")
    destino.guardar(df, batch_id)

consulta = encender_final(guardar_con_caida)
entregar_lotes(consulta, orden[PARADA:CAIDA - 1])
antes_de_la_caida = cifras(destino.leer())
caida["activa"] = True
entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, orden[CAIDA - 1])
try:
    consulta.processAllAvailable()
    fallo = ""
except Exception as e:
    fallo = str(e)
assert "Caída simulada" in fallo and not consulta.isActive, "La consulta debía fallar"
tras_la_caida = cifras(destino.leer())
assert tras_la_caida == antes_de_la_caida, "La tabla cambió aunque el paso 2 no se ejecutó"
print(f"Caída en el micro-lote {caida['micro_lote']} ({orden[CAIDA - 1]}): la consulta se detuvo con error; "
      f"la tabla sigue con {tras_la_caida['contratos']:,} contratos (sin ese lote) y cambios/ ya tiene la carpeta del micro-lote")

guardados_antes = len(destino.lotes)
consulta = encender_final(destino.guardar)
consulta.processAllAvailable()  # Spark repite el micro-lote que no quedó confirmado
repetido = next(b for b, n in destino.lotes[guardados_antes:] if n > 0)
assert repetido == caida["micro_lote"], "Al reiniciar no se repitió el micro-lote de la caída"
print(f"Reinicio: Spark repitió el micro-lote {repetido}; la tabla tiene ahora {cifras(destino.leer())['contratos']:,} contratos")
entregar_lotes(consulta, orden[CAIDA:])
consulta.stop()
tablas["A"] = foto()

Parada ordenada tras lote_08_regular: 212,108 contratos en la tabla


Caída en el micro-lote 27 (lote_15_regular): la consulta se detuvo con error; la tabla sigue con 346,176 contratos (sin ese lote) y cambios/ ya tiene la carpeta del micro-lote


Reinicio: Spark repitió el micro-lote 27; la tabla tiene ahora 372,530 contratos


### Escenario B: el flujo completo dos veces seguidas

Primero una ejecución completa. Después, dos formas de «ejecutarlo otra vez»:

1. **Con el mismo checkpoint:** se enciende la consulta de nuevo. Los 22 archivos ya están anotados y no se lee nada.
2. **Desde cero sobre el mismo destino:** se borran el checkpoint y la carpeta de entrada, se conserva `streaming/ventanas` y se entregan otra vez los 22 lotes. Spark lo procesa todo de nuevo y el destino reemplaza cada ventana.

In [18]:
# T3.4 · Escenario B: ejecución completa, reencendido con el mismo checkpoint y segunda ejecución desde cero sobre el mismo destino
empezar_de_cero()
consulta = encender_final(destino.guardar)
entregar_lotes(consulta, orden)
consulta.stop()
primera = cifras(destino.leer())

consulta = encender_final(destino.guardar)
consulta.processAllAvailable()
time.sleep(1)
releidas = sum(p["numInputRows"] for p in consulta.recentProgress)
consulta.stop()
assert releidas == 0 and cifras(destino.leer()) == primera, "Reencender con el mismo checkpoint cambió algo"
print(f"1) Reencendido con el mismo checkpoint: {releidas} filas leídas; cifras iguales")

empezar_de_cero(conservar_destino=True)
consulta = encender_final(destino.guardar)
entregar_lotes(consulta, orden)
time.sleep(1)
procesadas = sum(p["numInputRows"] for p in disparos_con_datos(consulta).values())
tablas["B"] = foto()
print(f"2) Segunda ejecución desde cero sobre el mismo destino: {procesadas:,} filas leídas otra vez")
display(pd.DataFrame([{"momento": "tras la primera ejecución", **primera}, {"momento": "tras la segunda ejecución", **cifras(tablas["B"])}]))

1) Reencendido con el mismo checkpoint: 0 filas leídas; cifras iguales


2) Segunda ejecución desde cero sobre el mismo destino: 530,192 filas leídas otra vez


,momento,ventanas,contratos,valor,huella
0,tras la primera ejecución,3735,529326,"175,067,420,085,034,807.00",358729028860944948943
1,tras la segunda ejecución,3735,529326,"175,067,420,085,034,807.00",358729028860944948943


### Escenario C: volver a depositar un lote ya procesado

Sobre la consulta del escenario B, que sigue encendida tras una ejecución completa, se vuelven a dejar en la entrada el último lote con el mismo nombre, el último lote con otro nombre y un lote antiguo con otro nombre (los tres casos de T3.3).

In [19]:
# T3.4 · Escenario C: tres lotes ya procesados vuelven a la carpeta de entrada
vistos = set(disparos_con_datos(consulta))
for archivo, nombre in [(orden[-1], None), (orden[-1], orden[-1] + "_copia"), (orden[4], orden[4] + "_copia")]:
    entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, archivo, nombre=nombre)
    consulta.processAllAvailable()
time.sleep(1)
releidas_c = sum(p["numInputRows"] for b, p in disparos_con_datos(consulta).items() if b not in vistos)
consulta.stop()
tablas["C"] = foto()
print(f"Lotes repetidos: Spark leyó {releidas_c:,} filas de los dos archivos con otro nombre y ninguna del archivo con el mismo nombre")
assert releidas_c == filas_lote[orden[-1]] + filas_lote[orden[4]]

Lotes repetidos: Spark leyó 44,633 filas de los dos archivos con otro nombre y ninguna del archivo con el mismo nombre


### Resultado de los tres escenarios

In [20]:
# T3.4 · Tabla de los tres escenarios frente a la ejecución limpia y al cálculo por lotes
ESCENARIOS = {
    "limpia": "Ejecución limpia (referencia)",
    "A": "A. Parada, reinicio y caída durante la escritura",
    "B": "B. Flujo completo dos veces seguidas",
    "C": "C. Lotes ya procesados depositados otra vez",
}
filas = []
for clave, nombre in ESCENARIOS.items():
    c = cifras(tablas[clave])
    d_limpia = ventanas_distintas(tablas[clave], tablas["limpia"])
    d_lotes = ventanas_distintas(tablas[clave], tablas["por lotes"])
    filas.append({
        "escenario": nombre, "ventanas": c["ventanas"], "contratos": c["contratos"], "valor total": c["valor"],
        "ventanas distintas de la ejecución limpia": d_limpia, "ventanas distintas del cálculo por lotes": d_lotes,
        "estado": "igual" if d_limpia == 0 and d_lotes == 0 and c == cifras(tablas["limpia"]) else "DISTINTO",
    })
c = cifras(tablas["por lotes"])
filas.append({"escenario": "Cálculo por lotes (T2.6)", "ventanas": c["ventanas"], "contratos": c["contratos"], "valor total": c["valor"],
              "ventanas distintas de la ejecución limpia": ventanas_distintas(tablas["por lotes"], tablas["limpia"]),
              "ventanas distintas del cálculo por lotes": 0, "estado": "referencia"})
tabla_escenarios = pd.DataFrame(filas)
display(tabla_escenarios)

# Resumen por ventana: contratos por año de adjudicación en cada escenario (cada año suma sus meses y departamentos)
por_anio = pd.DataFrame({
    ("por lotes" if k == "por lotes" else ESCENARIOS[k].split(".")[0].split(" (")[0]):
        t.groupBy(F.year("mes").alias("año")).agg(F.sum("contratos").alias("c")).toPandas().set_index("año")["c"]
    for k, t in tablas.items()
}).sort_index()
display(por_anio.style.format("{:,}"))

assert (tabla_escenarios["estado"] != "DISTINTO").all(), "Algún escenario no dio el mismo resultado"
assert (por_anio.nunique(axis=1) == 1).all(), "Algún año tiene cifras distintas entre escenarios"
print("Criterio de aceptación de T3.4: OK (en los tres escenarios el resultado es idéntico al de la ejecución limpia y al cálculo por lotes)")

,escenario,ventanas,contratos,valor total,ventanas distintas de la ejecución limpia,ventanas distintas del cálculo por lotes,estado
0,Ejecución limpia (referencia),3735,529326,"175,067,420,085,034,807.00",0,0,igual
1,"A. Parada, reinicio y caída durante la escritura",3735,529326,"175,067,420,085,034,807.00",0,0,igual
2,B. Flujo completo dos veces seguidas,3735,529326,"175,067,420,085,034,807.00",0,0,igual
3,C. Lotes ya procesados depositados otra vez,3735,529326,"175,067,420,085,034,807.00",0,0,igual
4,Cálculo por lotes (T2.6),3735,529326,"175,067,420,085,034,807.00",0,0,referencia


,Ejecución limpia,por lotes,A,B,C
año,,,,,
2015,29,29,29,29,29
2016,"1,859","1,859","1,859","1,859","1,859"
2017,"18,284","18,284","18,284","18,284","18,284"
2018,"32,141","32,141","32,141","32,141","32,141"
2019,"33,193","33,193","33,193","33,193","33,193"
2020,"38,294","38,294","38,294","38,294","38,294"
2021,"56,100","56,100","56,100","56,100","56,100"
2022,"66,717","66,717","66,717","66,717","66,717"
2023,"71,544","71,544","71,544","71,544","71,544"


Criterio de aceptación de T3.4: OK (en los tres escenarios el resultado es idéntico al de la ejecución limpia y al cálculo por lotes)


**Qué queda documentado para T3.5.**
- En los tres escenarios la tabla final tiene las mismas ventanas, con el mismo conteo y el mismo valor, que la ejecución limpia y que el cálculo por lotes.
- Si el flujo se cae mientras escribe (escenario A): el micro-lote no queda confirmado en `commits`, la tabla conserva las cifras anteriores a ese micro-lote y, al reiniciar, Spark lo repite con los mismos datos y el mismo número. El destino reescribe su carpeta de cambios y rehace los meses afectados.
- Entre la caída y el reinicio la tabla está atrasada (le falta el último lote), pero no tiene cifras duplicadas ni ventanas a medias en `tabla/`: quien la consulte en ese momento ve un estado anterior coherente.
- Ejecutar el flujo otra vez desde cero sobre el mismo destino (escenario B) no duplica, porque el destino reemplaza. Reencender con el mismo checkpoint no lee nada.
- La prueba simula la caída con un error dentro del destino, entre sus dos escrituras. No prueba la muerte del proceso de Spark ni un corte a mitad de la escritura de un archivo; en MinIO un objeto aparece completo o no aparece, y la escritura de particiones de Spark no es atómica entre particiones, así que el caso que queda cubierto por diseño (no por prueba) es el de una partición de `tabla/` reemplazada y otra no: la repetición del micro-lote rehace todas.